# ScienceQA — Inference Notebook

Loads a saved LoRA adapter and generates `submission.csv`.
Run this after the training notebook has saved an adapter.

**Adapter path:** set `ADAPTER_DIR` below to wherever the adapter was saved.
- Same Kaggle session: `./lora-final-seed42`
- Saved as Kaggle dataset output and re-attached: `/kaggle/input/<your-dataset>/lora-final-seed42`

In [ ]:
# ── 0. Install ────────────────────────────────────────────────────────────────
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate pillow

In [ ]:
# ── 1. Imports & Configuration ────────────────────────────────────────────────
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import PeftModel

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = Path("/kaggle/input/datasets/komvopoulos/finalexamdataset")
ADAPTER_DIR = "./lora-final-seed42"   # ← change if adapter is in a different location

# ── Model knobs (must match training notebook) ────────────────────────────────
MODEL_ID       = "HuggingFaceTB/SmolVLM-500M-Instruct"
IMG_SIZE       = 224
CHOICE_LETTERS = "ABCDEFGHIJ"

# ── Text-field limits (must match training notebook) ─────────────────────────
INCLUDE_LECTURE    = False
SOLUTION_MAX_CHARS = 400

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 2. Load CSVs ──────────────────────────────────────────────────────────────
val_df  = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

for df in [val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

TEST_HAS_SOLUTION = (
    "solution" in test_df.columns
    and test_df["solution"].notna().any()
    and test_df["solution"].astype(str).str.strip().ne("").any()
)
print(f"Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Solution available in test.csv: {TEST_HAS_SOLUTION}")

In [ ]:
# ── 2b. Locate test images ────────────────────────────────────────────────────
# Prints directory layout and probes candidate paths for the first test image.
# Read the output to find the correct root, then set IMAGE_ROOT below.
import os

print("── DATA_DIR contents ──────────────────────────────────────")
for entry in sorted(os.listdir(DATA_DIR)):
    p = DATA_DIR / entry
    tag = "DIR" if p.is_dir() else "file"
    print(f"  [{tag}]  {entry}")

images_dir = DATA_DIR / "images"
if images_dir.exists():
    print("\n── DATA_DIR/images/ contents ──────────────────────────────")
    for entry in sorted(os.listdir(images_dir)):
        p = images_dir / entry
        tag = "DIR" if p.is_dir() else "file"
        print(f"  [{tag}]  {entry}")

sample_path = test_df["image_path"].iloc[0]
print(f"\n── First test image_path value: '{sample_path}' ──────────")
candidates = [
    DATA_DIR / "images" / sample_path,
    DATA_DIR / sample_path,
    DATA_DIR / "images" / Path(sample_path).name,
    DATA_DIR / Path(sample_path).name,
]
for c in candidates:
    print(f"  {'EXISTS ✓' if c.exists() else 'missing  '}: {c}")

# ── Set IMAGE_ROOT to whichever prefix makes paths resolve ───────────────────
# After reading the output above, set this to one of:
#   DATA_DIR / "images"   (nested — worked for train/val)
#   DATA_DIR              (flat — try if nested is missing)
# The _open_image function below uses it automatically.
IMAGE_ROOT = DATA_DIR / "images"   # ← adjust if needed based on output above
print(f"\nIMAGE_ROOT set to: {IMAGE_ROOT}")

In [ ]:
# ── 3. Prompt builder (identical to training notebook) ────────────────────────
def _trunc(text, max_chars: int) -> str:
    text = str(text).strip()
    return text if len(text) <= max_chars else text[:max_chars] + "…"


def build_prompt(row, include_answer: bool = False, include_solution: bool = False) -> str:
    context_parts = []

    if INCLUDE_LECTURE:
        lecture = row.get("lecture", "") if isinstance(row, dict) else getattr(row, "lecture", "")
        if pd.notna(lecture) and str(lecture).strip():
            context_parts.append(_trunc(lecture, 500))

    hint = row.get("hint", "") if isinstance(row, dict) else getattr(row, "hint", "")
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())

    if include_solution:
        sol = row.get("solution", "") if isinstance(row, dict) else getattr(row, "solution", "")
        if pd.notna(sol) and str(sol).strip():
            context_parts.append("Reasoning: " + _trunc(sol, SOLUTION_MAX_CHARS))

    choices = row["choices"] if isinstance(row, dict) else row.choices
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    prompt = "<image>\n"
    if context_parts:
        prompt += "Context:\n" + "\n\n".join(context_parts) + "\n\n"
    question = row["question"] if isinstance(row, dict) else row.question
    prompt += f"Question: {question}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer = row["answer"] if isinstance(row, dict) else row.answer
        prompt += f" {CHOICE_LETTERS[int(answer)]}"

    return prompt

In [ ]:
# ── 4. Load saved adapter ─────────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(ADAPTER_DIR)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
# Force right-padding — must match the training collate_fn setting.
# Idefics3 may save/load with padding_side="left" (generation default).
processor.tokenizer.padding_side = "right"

print(f"padding_side : {processor.tokenizer.padding_side}")
print(f"pad_token_id : {processor.tokenizer.pad_token_id}  ({processor.tokenizer.pad_token!r})")
print(f"eos_token_id : {processor.tokenizer.eos_token_id}  ({processor.tokenizer.eos_token!r})")

base = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
print(f"Adapter loaded from {ADAPTER_DIR}")

In [ ]:
# ── 5. Log-likelihood scoring ─────────────────────────────────────────────────
# IMAGE_ROOT is set in cell 2b — update it there if images aren't found.

def _open_image(image_path: str, img_size: int) -> Image.Image:
    path = IMAGE_ROOT / image_path
    if not path.exists():
        return Image.new("RGB", (img_size, img_size), color=(128, 128, 128))
    return Image.open(path).convert("RGB").resize((img_size, img_size), Image.BICUBIC)


@torch.inference_mode()
def score_choices(model, processor, image: Image.Image, row: dict) -> list:
    """Return log P(answer_token | context) for each answer choice."""
    choices     = row["choices"]
    n           = len(choices)
    prompt_base = build_prompt(row, include_answer=False,
                               include_solution=TEST_HAS_SOLUTION)

    full_texts = [prompt_base + f" {CHOICE_LETTERS[i]}" for i in range(n)]

    # Force right-padding every call — defensive against anything that might
    # flip padding_side back to "left" between batches.
    processor.tokenizer.padding_side = "right"

    inputs = processor(
        text=full_texts,
        images=[image] * n,
        return_tensors="pt",
        padding=True,
    )
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v
              for k, v in inputs.items()}

    eos_id = processor.tokenizer.eos_token_id
    pad_id = processor.tokenizer.pad_token_id
    logits = model(**inputs).logits  # (n, seq_len, vocab_size)

    log_probs = []
    for i in range(n):
        seq = inputs["input_ids"][i].tolist()

        # Walk backward from the very last token, skipping EOS and PAD.
        # Using len(seq)-1 (not attention_mask.sum()-1) works for both
        # right-padded and left-padded sequences.
        answer_pos = len(seq) - 1
        while answer_pos >= 0 and seq[answer_pos] in (eos_id, pad_id):
            answer_pos -= 1

        pred_pos        = answer_pos - 1
        answer_token_id = seq[answer_pos]

        lp = torch.log_softmax(logits[i, pred_pos], dim=-1)[answer_token_id].item()
        log_probs.append(lp)

    return log_probs


def predict_one(model, processor, image: Image.Image, row: dict) -> int:
    lps = score_choices(model, processor, image, row)
    return int(torch.tensor(lps).argmax())


def run_inference(model, processor, df: pd.DataFrame,
                  img_size: int = IMG_SIZE,
                  desc: str = "Inference") -> tuple:
    """Predict all rows in df. Returns (list_of_preds, accuracy_or_None)."""
    model.eval()
    preds, correct, total, missing = [], 0, 0, 0

    # ── Pre-flight diagnostic ──────────────────────────────────────────────────
    # Prints raw log_probs for the first 5 samples so you can verify the
    # scoring function is working BEFORE waiting for all 1008 predictions.
    print("=== Pre-flight diagnostic (first 5 samples) ===")
    _all_equal = True
    for _di in range(min(5, len(df))):
        _dr  = df.iloc[_di].to_dict()
        _img = _open_image(_dr["image_path"], img_size)
        _lps = score_choices(model, processor, _img, _dr)
        _p   = int(torch.tensor(_lps).argmax())
        _gt  = _dr.get("answer", -1)
        _gt_str = f"  gt={CHOICE_LETTERS[int(_gt)]}" if pd.notna(_gt) and int(_gt) >= 0 else ""
        print(f"  [{_di}] n_choices={len(_lps)}  "
              f"lps={[round(x, 3) for x in _lps]}  "
              f"pred={CHOICE_LETTERS[_p]}{_gt_str}")
        if len(set(round(x, 4) for x in _lps)) > 1:
            _all_equal = False
    if _all_equal:
        print("  WARNING: all log_probs are identical across every sample checked.")
        print("  This means score_choices has a bug — all predictions will be 0.")
    else:
        print("  OK: log_probs show variation — scoring function is working.")
    print("=== End diagnostic ===\n")
    # ──────────────────────────────────────────────────────────────────────────

    bar = tqdm(df.iterrows(), total=len(df), desc=desc, dynamic_ncols=True)
    for _, row in bar:
        img = _open_image(row["image_path"], img_size)
        if img.getpixel((0, 0)) == (128, 128, 128):
            missing += 1
        pred = predict_one(model, processor, img, row.to_dict())
        preds.append(pred)

        gt = row.get("answer", -1)
        if pd.notna(gt) and int(gt) >= 0:
            correct += (pred == int(gt))
            total   += 1
            bar.set_postfix(acc=f"{correct}/{total} ({correct/total:.3f})")

    if missing:
        print(f"Note: {missing}/{len(df)} rows used grey placeholder (no image file).")
    acc = correct / total if total > 0 else None
    if acc is not None:
        print(f"Final accuracy: {correct}/{total} = {acc:.4f}")
    return preds, acc

In [ ]:
# ── 7. Predict test set and write submission.csv ───────────────────────────────
test_preds, _ = run_inference(model, processor, test_df, desc="Test")

submission = pd.DataFrame({
    "id":     test_df["id"],
    "answer": test_preds,
})
submission.to_csv("submission.csv", index=False)
print(f"submission.csv written — {len(submission)} rows")
print(submission.head(10))

In [ ]:
# ── 8. Auto-download submission.csv ───────────────────────────────────────────
import base64
from IPython.display import HTML, display

with open("submission.csv", "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

display(HTML(f"""
<script>
(function() {{
    var a = document.createElement('a');
    a.href = 'data:text/csv;base64,{b64}';
    a.download = 'submission.csv';
    document.body.appendChild(a);
    a.click();
    document.body.removeChild(a);
}})();
</script>
<p>Download triggered. If nothing happened, <a href="data:text/csv;base64,{b64}" download="submission.csv">click here</a>.</p>
"""))